In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

# ================================
#  LOAD DATA
# ================================
df = pd.read_csv('Tweets.csv')

# retain only required columns
df = df[['text', 'airline_sentiment']]

# ================================
#  PREPROCESS
# ================================
# remove neutral → binary classification
df = df[df['airline_sentiment'] != 'neutral']

# map labels
df['label'] = df['airline_sentiment'].map({'positive': 1, 'negative': 0})

# basic cleaning
df['text'] = df['text'].str.lower()

# ================================
#  BALANCE DATA
# ================================
df_pos = df[df['label'] == 1]
df_neg = df[df['label'] == 0]

# downsample majority class
if len(df_pos) > len(df_neg):
    df_pos = resample(df_pos, replace=False, n_samples=len(df_neg), random_state=42)
else:
    df_neg = resample(df_neg, replace=False, n_samples=len(df_pos), random_state=42)

df_balanced = pd.concat([df_pos, df_neg])

# shuffle data
df_balanced = df_balanced.sample(frac=1, random_state=42)

# ================================
#  TRAIN TEST SPLIT
# ================================
X = df_balanced['text']
y = df_balanced['label']

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
#  TF-IDF VECTORIZATION
# ================================
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

Xtrain_vec = vectorizer.fit_transform(Xtrain)
Xtest_vec = vectorizer.transform(Xtest)

# ================================
#  LOGISTIC REGRESSION MODEL
# ================================
model = LogisticRegression(max_iter=1000)
model.fit(Xtrain_vec, ytrain)

# predictions
y_pred = model.predict(Xtest_vec)

# ================================
#  EVALUATION
# ================================
print("Accuracy:", accuracy_score(ytest, y_pred))
print("\nClassification Report:\n", classification_report(ytest, y_pred))

# ================================
#  IMPORTANT WORDS (FEATURE ANALYSIS)
# ================================
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

# top positive words
top_pos_idx = np.argsort(coefficients)[-10:]
top_neg_idx = np.argsort(coefficients)[:10]

print("\nTop Positive Words:")
for i in reversed(top_pos_idx):
    print(feature_names[i])

print("\nTop Negative Words:")
for i in top_neg_idx:
    print(feature_names[i])

# ================================
#  WRONG PREDICTIONS
# ================================
Xtest_df = pd.DataFrame({
    'text': Xtest,
    'true': ytest,
    'pred': y_pred
})

wrong = Xtest_df[Xtest_df['true'] != Xtest_df['pred']]

print("\nTotal Wrong Predictions:", len(wrong))

print("\nSample Wrong Predictions:")
print(wrong.head(5))

# ================================
#  LABEL CONTRADICTION ANALYSIS
# ================================
positive_words = set(feature_names[top_pos_idx])
negative_words = set(feature_names[top_neg_idx])

wrong_pos_as_neg = 0
wrong_neg_as_pos = 0

for _, row in df.iterrows():
    words = set(row['text'].split())

    # positive words but labeled negative
    if row['label'] == 0 and len(words & positive_words) > 0:
        wrong_pos_as_neg += 1

    # negative words but labeled positive
    if row['label'] == 1 and len(words & negative_words) > 0:
        wrong_neg_as_pos += 1

print("\nTweets with +ve words but labeled negative:", wrong_pos_as_neg)
print("Tweets with -ve words but labeled positive:", wrong_neg_as_pos)

Accuracy: 0.8742071881606766

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.89      0.88       486
           1       0.88      0.86      0.87       460

    accuracy                           0.87       946
   macro avg       0.87      0.87      0.87       946
weighted avg       0.87      0.87      0.87       946


Top Positive Words:
thanks
thank
great
awesome
southwestair
love
amazing
virginamerica
jetblue
good

Top Negative Words:
hold
hours
worst
delayed
cancelled
bag
need
hour
told
flightled

Total Wrong Predictions: 119

Sample Wrong Predictions:
                                                   text  true  pred
1087  @united counter agents at rdu deserve a medal....     1     0
1315  @united have michelle at t1 ord train your oth...     1     0
663   @united private jet would have been cool! do d...     1     0
3434  @united #orthodoc on-call to in-flight! glad t...     1     0
5833  @southwestair lets chat about 